# SAR + Optical Fusion — Colab Training
**Roham Rasouli Kerahroudi — Sakarya Universitesi Bitirme Projesi**

Her sessionda sadece Cell 7'yi calistir — Drive'daki checkpoint'ten otomatik devam eder.

**Ilk kez:** Cell 1 > 2 > 3 > 4 > 5 > 6 > 7

**Sonraki session:** Cell 1 > 7

In [ ]:
# CELL 1 — GPU kontrol + Google Drive bagla
import torch
assert torch.cuda.is_available(), 'GPU yok! Runtime > Change runtime type > T4'
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = '/content/drive/MyDrive/sar_fusion_runs'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoint dizini: {CKPT_DIR}')


In [ ]:
# CELL 2 — Kaggle credentials
from google.colab import files
import os

uploaded = files.upload()  # kaggle.json sec

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'wb') as f:
    f.write(list(uploaded.values())[0])
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
print('Kaggle credentials OK')


In [ ]:
# CELL 3 — M4-SAR dataset indir (~10-15 dakika)
import os
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

!pip install kaggle -q
!kaggle datasets download -d wchao0601/m4-sar -p {DATA_DIR} --unzip -q

print('Indirme tamamlandi. Klasor yapisi:')
!find {DATA_DIR} -maxdepth 5 -type d | sort | head -30


In [ ]:
# CELL 4 — DATA_ROOT otomatik bul
import subprocess
from pathlib import Path

DATA_DIR = '/content/data'
result = subprocess.run(['find', DATA_DIR, '-type', 'd', '-name', 'train'],
                        capture_output=True, text=True)
train_dirs = [d for d in result.stdout.strip().split('\n') if d]

print('Bulunan train klasorleri:')
for d in train_dirs:
    print(f'  {d}')

if train_dirs:
    parents = sorted(set(str(Path(d).parent.parent) for d in train_dirs))
    DATA_ROOT = parents[0]
else:
    DATA_ROOT = DATA_DIR

print(f'\nDATA_ROOT = {DATA_ROOT}')
print('Eger yanlis gorunuyorsa Cell 7deki DATA_ROOT satirini manuel duzenle')


In [ ]:
# CELL 5 — GitHub'dan kodu cek
%cd /content
!git clone https://github.com/RohamRasouli/sar-optical-fusion.git
%cd sar-optical-fusion
!git log --oneline -5


In [ ]:
# CELL 6 — Bagimliliklar kur
!pip install rasterio -q
!pip install -e . -q
print('Kurulum tamam')


In [ ]:
# CELL 7a — Dizin hazirla (session yenilenince buradan basla)
import os

CKPT_DIR = '/content/drive/MyDrive/sar_fusion_runs'
os.makedirs(CKPT_DIR, exist_ok=True)
os.chdir('/content/sar-optical-fusion')

best_ckpt = f'{CKPT_DIR}/best.pt'
if os.path.exists(best_ckpt):
    print(f'[RESUME] {best_ckpt} mevcut — Cell 7b komutu --resume iceriyor')
else:
    print('[FRESH] Checkpoint yok — sifirdan baslanacak')


In [ ]:
# CELL 7b — EGITIM (! ile dogrudan shell — output canli akar)
# Resume icin: --resume /content/drive/MyDrive/sar_fusion_runs/best.pt ekle
!python -m src.train \
  --config configs/kaggle_p100.yaml \
  --data_root "/content/data/M4-SAR/M4-SAR" \
  --output "/content/drive/MyDrive/sar_fusion_runs"


In [ ]:
# CELL 8 — Eval (egitim bittikten sonra)
import os

CKPT_DIR  = '/content/drive/MyDrive/sar_fusion_runs'
DATA_ROOT = '/content/data/M4-SAR/M4-SAR'

print('Checkpointler:')
!ls -lh {CKPT_DIR}

best_ckpt = f'{CKPT_DIR}/best.pt'
eval_cmd = f'python -m src.eval --checkpoint "{best_ckpt}" --config configs/kaggle_p100.yaml --data_root "{DATA_ROOT}" --batch_size 4 --max_samples 1000'
print(eval_cmd)
!{eval_cmd}
